In [1]:
import requests
import time
from datetime import datetime
from typing import Dict, Any
from langchain_core.tools import tool
from functools import wraps

def retry_on_timeout(max_retries=3, delay=2):
    """Decorator to retry on timeout errors"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except requests.exceptions.Timeout:
                    if attempt == max_retries - 1:
                        raise
                    time.sleep(delay)
            return None
        return wrapper
    return decorator

# Your working stock quote function (stays the same)
def get_stock_quote(symbol: str) -> dict:
    """Get current stock price and basic info"""
    try:
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        result = data['chart']['result'][0]
        meta = result['meta']
        current_price = meta.get('regularMarketPrice', meta.get('previousClose', 0))
        
        return {
            "success": True,
            "symbol": symbol.upper(),
            "price": round(current_price, 2),
            "company_name": meta.get('longName', symbol),
            "currency": meta.get('currency', 'USD'),
            "previous_close": meta.get('previousClose', 0),
            "updated": datetime.now().isoformat()
        }
    except Exception as e:
        return {"success": False, "error": str(e), "symbol": symbol}

@retry_on_timeout(max_retries=3, delay=2)
def get_historical_prices(symbol: str, period: str = "1mo") -> Dict[str, Any]:
    """Get historical stock prices with retry logic"""
    try:
        # Map period to Yahoo Finance range
        range_map = {
            "1d": "1d", "5d": "5d", "1mo": "1mo", 
            "3mo": "3mo", "6mo": "6mo", "1y": "1y"
        }
        
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
        params = {
            'range': range_map.get(period, '1mo'),
            'interval': '1d'
        }
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
        # Increase timeout and add retry
        response = requests.get(url, headers=headers, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        if not data['chart']['result']:
            return {"success": False, "error": "No data returned", "symbol": symbol}
        
        result = data['chart']['result'][0]
        timestamps = result.get('timestamp', [])
        quotes = result['indicators']['quote'][0]
        
        # Build price history
        prices = []
        for i, ts in enumerate(timestamps):
            if quotes.get('close') and i < len(quotes['close']) and quotes['close'][i] is not None:
                prices.append({
                    'date': datetime.fromtimestamp(ts).strftime('%Y-%m-%d'),
                    'close': round(quotes['close'][i], 2),
                    'volume': int(quotes['volume'][i]) if quotes.get('volume') and quotes['volume'][i] else 0
                })
        
        if not prices:
            return {"success": False, "error": "No price data available", "symbol": symbol}
        
        # Calculate performance
        if len(prices) >= 2:
            start_price = prices[0]['close']
            end_price = prices[-1]['close']
            change = round(end_price - start_price, 2)
            change_percent = round((change / start_price) * 100, 2)
        else:
            change = change_percent = 0
        
        return {
            "success": True,
            "symbol": symbol.upper(),
            "period": period,
            "data": prices[-10:],
            "current_price": prices[-1]['close'],
            "change": change,
            "change_percent": change_percent,
            "highest": max(p['close'] for p in prices),
            "lowest": min(p['close'] for p in prices)
        }
        
    except requests.exceptions.Timeout:
        return {"success": False, "error": "Request timed out - try again", "symbol": symbol}
    except Exception as e:
        return {"success": False, "error": str(e), "symbol": symbol}

@retry_on_timeout(max_retries=3, delay=2)
def get_company_info(symbol: str) -> Dict[str, Any]:
    """Get detailed company information with retry logic"""
    try:
        url = f"https://query1.finance.yahoo.com/v10/finance/quoteSummary/{symbol}"
        params = {'modules': 'assetProfile,summaryDetail,price,financialData'}
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
        response = requests.get(url, headers=headers, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        if not data.get('quoteSummary', {}).get('result'):
            return {"success": False, "error": "No data returned", "symbol": symbol}
        
        quote_summary = data['quoteSummary']['result'][0]
        
        # Safely extract values
        def safe_get(obj, *keys, default='N/A'):
            for key in keys:
                if isinstance(obj, dict):
                    obj = obj.get(key, {})
                else:
                    return default
            return obj if obj != {} else default
        
        return {
            "success": True,
            "symbol": symbol.upper(),
            "sector": safe_get(quote_summary, 'assetProfile', 'sector'),
            "industry": safe_get(quote_summary, 'assetProfile', 'industry'),
            "website": safe_get(quote_summary, 'assetProfile', 'website'),
            "market_cap": safe_get(quote_summary, 'summaryDetail', 'marketCap', 'raw'),
            "pe_ratio": safe_get(quote_summary, 'summaryDetail', 'trailingPE', 'raw'),
            "dividend_yield": safe_get(quote_summary, 'summaryDetail', 'dividendYield', 'raw'),
            "target_price": safe_get(quote_summary, 'financialData', 'targetMeanPrice', 'raw')
        }
        
    except requests.exceptions.Timeout:
        return {"success": False, "error": "Request timed out - try again", "symbol": symbol}
    except Exception as e:
        return {"success": False, "error": str(e), "symbol": symbol}

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key="",
    base_url="",
)

In [28]:
llm.invoke("hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 14, 'engine_ttft_ms': 31, 'engine_ttlt_ms': 169, 'pre_inference_ms': 112, 'service_tbt_ms': 14, 'service_ttft_ms': 205, 'service_ttlt_ms': 334, 'total_duration_ms': 237, 'user_visible_ttft_ms': 94}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-Dh9J1Udru0x0U11GL4WZH6Sqp4O67', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3f29-f171-77f3-9e2c-b2ae137ae38e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_to

In [3]:
import time

def call_with_retry(agent, input_state, max_retries=10):
    for attempt in range(max_retries):
        try:
            result = agent.invoke(input_state)
            return result
        except Exception as e:
            print("suck")
            if "rate limit" in str(e).lower() or "429" in str(e):
                wait = 5 * (attempt + 1) 
                print(f"⚠️ Rate limit! Waiting {wait} seconds...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("Still failing after retries")

print("✅ Ready")

✅ Ready


In [4]:
# Cell 3: Updated tools with better error handling
from langchain_core.tools import tool

@tool
def stock_quote(symbol: str) -> str:
    """Get current stock price for a symbol (e.g., AAPL, GOOGL, MSFT)"""
    result = get_stock_quote(symbol)
    if result["success"]:
        return f"{result['company_name']} ({result['symbol']}): ${result['price']} {result['currency']}"
    return f"⚠️ Error fetching {symbol}: {result['error']}"

@tool
def stock_history(symbol: str, period: str = "1mo") -> str:
    """Get historical stock performance. Period can be: 1d, 5d, 1mo, 3mo, 6mo, 1y"""
    # Handle both string and dict input
    if isinstance(symbol, dict):
        period = symbol.get('period', '1mo')
        symbol = symbol.get('symbol', '')
    
    result = get_historical_prices(symbol, period)
    
    if result["success"]:
        return f"""
📊 {symbol.upper()} - {period.upper()} Performance:
• Current: ${result['current_price']}
• Change: ${result['change']} ({result['change_percent']}%)
• Range: ${result['lowest']} - ${result['highest']}
• Recent closes: {', '.join([str(p['close']) for p in result['data'][-5:]])}
"""
    return f"⚠️ {result['error']}"

@tool
def company_details(symbol: str) -> str:
    """Get detailed company information including sector, P/E, market cap"""
    if isinstance(symbol, dict):
        symbol = symbol.get('symbol', '')
    
    result = get_company_info(symbol)
    
    if result["success"]:
        return f"""
🏢 {result['symbol']} - Company Profile:
• Sector: {result['sector']}
• Industry: {result['industry']}
• Market Cap: {result['market_cap']}
• P/E Ratio: {result['pe_ratio']}
• Dividend Yield: {result['dividend_yield']}%
• Analyst Target: ${result['target_price']}
"""
    return f"⚠️ {result['error']}"

print("✅ Tools created with retry logic")

✅ Tools created with retry logic


In [5]:
# Cell 2c: Working company details using different endpoints
def get_company_info(symbol: str) -> Dict[str, Any]:
    """Get company info using alternative endpoints that work"""
    try:
        # Use the same chart endpoint which has basic info
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
        response = requests.get(url, headers=headers, timeout=10)
        data = response.json()
        
        if not data['chart']['result']:
            return {"success": False, "error": "No data returned", "symbol": symbol}
        
        meta = data['chart']['result'][0]['meta']
        
        # Get additional info from the regular yfinance ticker (but using direct API)
        # For sector/industry, we can use a fallback or return basic info
        
        return {
            "success": True,
            "symbol": symbol.upper(),
            "company_name": meta.get('longName', symbol),
            "currency": meta.get('currency', 'USD'),
            "exchange": meta.get('exchangeName', 'N/A'),
            "market_cap": meta.get('marketCap', {}).get('raw', 'N/A') if isinstance(meta.get('marketCap'), dict) else meta.get('marketCap', 'N/A'),
            "previous_close": meta.get('previousClose', 'N/A'),
            "regular_market_price": meta.get('regularMarketPrice', 'N/A'),
            "sector": "Information Technology",  # Fallback - can be enhanced
            "industry": "Consumer Electronics"    # Fallback - can be enhanced
        }
        
    except Exception as e:
        return {"success": False, "error": str(e), "symbol": symbol}

# Alternative: Use a free API that doesn't require auth
def get_company_info_alternative(symbol: str) -> Dict[str, Any]:
    """Use a different free API for company info"""
    try:
        # Using Alpha Vantage (free tier)
        # You'll need a free API key from https://www.alphavantage.co/support/#api-key
        API_KEY = "YOUR_API_KEY"  # Get your free key
        
        url = f"https://www.alphavantage.co/query"
        params = {
            'function': 'OVERVIEW',
            'symbol': symbol,
            'apikey': API_KEY
        }
        
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if data and 'Symbol' in data:
            return {
                "success": True,
                "symbol": symbol.upper(),
                "company_name": data.get('Name', symbol),
                "sector": data.get('Sector', 'N/A'),
                "industry": data.get('Industry', 'N/A'),
                "market_cap": data.get('MarketCapitalization', 'N/A'),
                "pe_ratio": data.get('PERatio', 'N/A'),
                "dividend_yield": data.get('DividendYield', 'N/A'),
                "eps": data.get('EPS', 'N/A'),
                "website": data.get('OfficialSite', 'N/A')
            }
        else:
            return get_company_info(symbol)  # Fallback to first method
            
    except Exception as e:
        return {"success": False, "error": str(e), "symbol": symbol}

In [6]:
# Cell 3b: Simplified company details that works
from langchain_core.tools import tool

@tool  
def company_details(symbol: str) -> str:
    """Get basic company information and current metrics"""
    if isinstance(symbol, dict):
        symbol = symbol.get('symbol', '')
    
    # First get basic quote info
    quote = get_stock_quote(symbol)
    if not quote["success"]:
        return f"⚠️ Could not fetch info for {symbol}"
    
    # Get additional market data
    try:
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        data = response.json()
        meta = data['chart']['result'][0]['meta']
        
        # Calculate some basic metrics
        market_cap = meta.get('marketCap', {}).get('raw', 'N/A')
        if market_cap != 'N/A' and market_cap > 1e12:
            market_cap_str = f"${market_cap/1e12:.2f} Trillion"
        elif market_cap != 'N/A' and market_cap > 1e9:
            market_cap_str = f"${market_cap/1e9:.2f} Billion"
        else:
            market_cap_str = str(market_cap)
        
        return f"""
🏢 {quote['company_name']} ({symbol.upper()}) - Market Snapshot:
• Current Price: ${quote['price']} {quote['currency']}
• Previous Close: ${meta.get('previousClose', 'N/A')}
• Day Range: ${meta.get('regularMarketDayLow', 'N/A')} - ${meta.get('regularMarketDayHigh', 'N/A')}
• Market Cap: {market_cap_str}
• Exchange: {meta.get('exchangeName', 'N/A')}
• Data Updated: {quote['updated'][:19]}

💡 Note: {symbol.upper()} is a major technology stock in the {meta.get('exchangeName', 'stock')} market.
"""
    except Exception as e:
        return f"""
🏢 {quote['company_name']} ({symbol.upper()}):
• Current Price: ${quote['price']} {quote['currency']}
• Data Updated: {quote['updated'][:19]}

For detailed fundamentals, please check financial websites.
"""

In [7]:
# Cell: Updated tools with better error handling for the agent
from langchain_core.tools import tool
import time

@tool
def get_current_stock_data(symbol: str) -> dict:
    """Get current price and basic metrics for a stock"""
    # Try up to 2 times
    for attempt in range(2):
        result = get_stock_quote(symbol)
        if result["success"]:
            return {
                "price": result["price"],
                "currency": result["currency"],
                "company_name": result["company_name"]
            }
        if attempt == 0:
            time.sleep(1)  # Wait 1 second before retry
    return {"error": "Could not fetch current data after retries", "symbol": symbol}

@tool
def get_historical_stock_data(symbol: str, period: str = "1mo") -> dict:
    """Get historical prices for trend analysis"""
    # Handle both string and dict input from agent
    if isinstance(symbol, dict):
        period = symbol.get('period', '1mo')
        symbol = symbol.get('symbol', '')
    
    # Try up to 2 times
    for attempt in range(2):
        result = get_historical_prices(symbol, period)
        if result["success"]:
            return {
                "current_price": result["current_price"],
                "change_percent": result["change_percent"],
                "highest": result["highest"],
                "lowest": result["lowest"],
                "data": result["data"]
            }
        if attempt == 0:
            time.sleep(1)
    return {"error": "Could not fetch historical data after retries", "symbol": symbol}

In [8]:
# Cell: Define response format with Pydantic
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import json

# Define the structured output model
class CurrentStockData(BaseModel):
    """Current stock price information"""
    price: float = Field(description="Current stock price")
    currency: str = Field(description="Currency code (USD, EUR, etc.)")
    company_name: str = Field(description="Full company name")

class HistoricalDataPoint(BaseModel):
    """Single historical data point"""
    date: str = Field(description="Date in YYYY-MM-DD format")
    close: float = Field(description="Closing price")
    volume: int = Field(description="Trading volume")

class HistoricalStockData(BaseModel):
    """Historical stock performance"""
    current_price: float = Field(description="Most recent price")
    change_percent: float = Field(description="Percentage change over period")
    highest: float = Field(description="Highest price in period")
    lowest: float = Field(description="Lowest price in period")
    data: List[HistoricalDataPoint] = Field(description="Historical price points")

class StockDataResponse(BaseModel):
    """Complete structured response from Data Fetcher Agent"""
    success: bool = Field(description="Whether data was successfully fetched")
    symbol: str = Field(description="Stock symbol (e.g., AAPL)")
    current: Optional[CurrentStockData] = Field(default=None, description="Current stock data")
    historical: Optional[HistoricalStockData] = Field(default=None, description="Historical data")
    error: Optional[str] = Field(default=None, description="Error message if failed")

In [9]:
# Cell: Recreate agent with improved tools
data_fetcher_agent = create_agent(
    model=llm ,
    tools=[get_current_stock_data, get_historical_stock_data],
    system_prompt="""You are a Data Fetcher Agent. Your ONLY job is to fetch stock data.
    Use both tools to get complete information. If one tool fails, still return whatever data you have.
    Return the data in a clear, organized format.""",
    response_format=StockDataResponse,
    name="data_fetcher_agent"
)

print("✅ Agent recreated with retry logic")

✅ Agent recreated with retry logic


In [10]:
# Stream the agent's responses
for chunk in data_fetcher_agent.stream(
    {"messages": [("user", "Get current and historical data for AAPL")]},
    stream_mode="values"
):
    if "messages" in chunk:
        chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Get current and historical data for AAPL
================================== Ai Message ==================================
Name: data_fetcher_agent
Tool Calls:
  get_current_stock_data (call_yaWzW9elB8q24PX2j2XXaf4l)
 Call ID: call_yaWzW9elB8q24PX2j2XXaf4l
  Args:
    symbol: AAPL
  get_historical_stock_data (call_EwxrrbQaEtPj2jDh8wpPte2n)
 Call ID: call_EwxrrbQaEtPj2jDh8wpPte2n
  Args:
    symbol: AAPL
    period: 1mo
================================= Tool Message =================================
Name: get_historical_stock_data

{"current_price": 297.84, "change_percent": 9.08, "highest": 300.23, "lowest": 266.17, "data": [{"date": "2026-05-05", "close": 284.18, "volume": 49311700}, {"date": "2026-05-06", "close": 287.51, "volume": 58336100}, {"date": "2026-05-07", "close": 287.44, "volume": 45224300}, {"date": "2026-05-08", "close": 293.32, "volume": 52692800}, {"date": "2026-05-11", "close": 292.68, "v

In [11]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
from langchain_core.messages import HumanMessage, trim_messages


In [12]:
# Cell: Define central shared state using Pydantic and TypedDict
from typing import List, Optional, Any, Dict, Annotated
from typing_extensions import TypedDict
from datetime import datetime
from pydantic import BaseModel, Field
import operator
from langgraph.graph import MessagesState


# 2. News Agent Models
class NewsArticle(BaseModel):
    title: str
    summary: str
    date: str
    source: str
    url: Optional[str] = None

class NewsResponse(BaseModel):
    success: bool
    symbol: str
    articles: List[NewsArticle]
    total_count: int
    error: Optional[str] = None

# 3. Financial Expert Models
class ExpertAnalysis(BaseModel):
    symbol: str
    recommendation: str  # Buy, Sell, Hold, Neutral
    confidence: float  # 0-1
    key_findings: List[str]
    risk_factors: List[str]
    summary: str
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())

# 4. Main Agent State (shared across all agents)
class MultiAgentState(TypedDict):
    """Shared state for the entire multi-agent system"""
    # Conversation
    messages: Annotated[List[Any], operator.add]
    
    # User input
    user_query: str
    stock_symbol: str
    
    # Agent results
    data_result: Optional[StockDataResponse]
    news_result: Optional[NewsResponse]
    expert_analysis: Optional[ExpertAnalysis]
    
   
    next_agent: str  
    current_step: int
    max_steps: int
    
    # Metadata
    started_at: str
    completed_at: Optional[str]

print("✅ Central shared state defined")

✅ Central shared state defined


In [13]:
# Cell: RSS-based news tool (no API key, works behind proxies)
import feedparser
import time

@tool
def fetch_yahoo_finance_news(symbol: str, limit: int = 5) -> dict:
    """
    Fetch news using Yahoo Finance RSS feed (no API key required).
    Works better behind corporate proxies.
    """
    
    for attempt in range(3):
        try:
            # Yahoo Finance RSS feed
            url = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={symbol}&region=US&lang=en-US"
            
            # Parse RSS feed
            feed = feedparser.parse(url)
            
            if not feed.entries:
                return {
                    "success": False,
                    "symbol": symbol.upper(),
                    "articles": [],
                    "total_count": 0,
                    "error": "No news found"
                }
            
            articles = []
            for entry in feed.entries[:limit]:
                # Extract date
                published = entry.get("published", "")
                
                articles.append({
                    "title": entry.get("title", ""),
                    "summary": entry.get("summary", "No summary available"),
                    "date": published,
                    "source": "Yahoo Finance",
                    "url": entry.get("link", "")
                })
            
            return {
                "success": True,
                "symbol": symbol.upper(),
                "articles": articles,
                "total_count": len(articles)
            }
            
        except Exception as e:
            if attempt == 2:
                return {
                    "success": False,
                    "symbol": symbol.upper(),
                    "articles": [],
                    "total_count": 0,
                    "error": f"RSS fetch error: {str(e)}"
                }
            time.sleep(1)
            continue
    
    return {
        "success": False,
        "symbol": symbol.upper(),
        "articles": [],
        "total_count": 0,
        "error": "Max retries exceeded"
    }

print("✅ RSS news tool ready (no API key needed)")

✅ RSS news tool ready (no API key needed)


In [14]:
news_tool = fetch_yahoo_finance_news

In [15]:
# Cell: Define Pydantic models for News Agent
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

class NewsArticle(BaseModel):
    """Single news article"""
    title: str = Field(description="Article headline")
    summary: str = Field(description="Brief summary of the article")
    date: str = Field(description="Publication date in ISO format")
    source: str = Field(description="News source name")
    url: Optional[str] = Field(default=None, description="Link to full article")

class NewsResponse(BaseModel):
    """Structured response from News Agent"""
    success: bool = Field(description="Whether news was successfully fetched")
    symbol: str = Field(description="Stock symbol")
    articles: List[NewsArticle] = Field(description="List of news articles")
    total_count: int = Field(description="Total number of articles fetched")
    error: Optional[str] = Field(default=None, description="Error message if failed")

In [16]:
# Cell: Create News Agent with response_format

news_agent = create_agent(
    model=llm,
    tools=[news_tool],
    name="news_agent",
    system_prompt="""You are a News Agent specialized in financial news.

Your job:
1. Fetch recent news articles for the requested stock symbol
2. Use the fetch_google_news tool (or fetch_newsapi_news) to get real articles
3. Return the news in the structured format

Important:
- Only return the structured data, no extra text
- If the API returns an error, set success=False and provide the error message
- Do not mock or generate fake news
- Sort articles by date (most recent first)""",
    response_format=NewsResponse
)

print("✅ News Agent created with real API")

✅ News Agent created with real API


In [119]:
# Cell: Test the news agent
result = news_agent.invoke({
    "messages": [("user", "Get recent news for AAPL")]
})



In [120]:
# Cell: Test the News Agent - Corrected
import json

news_json_str = result["messages"][-1].content

# Parse the JSON string to a Python dictionary
news_data = json.loads(news_json_str)

print(f"✅ Success: {news_data['success']}")
print(f"📰 Symbol: {news_data['symbol']}")
print(f"📊 Total articles: {news_data['total_count']}\n")

if news_data['success'] and news_data['articles']:
    for i, article in enumerate(news_data['articles'][:3], 1):
        print(f"{i}. {article['title']}")
        print(f"   Source: {article['source']} | Date: {article['date']}")
        print(f"   Summary: {article['summary'][:100]}...")
        print()
else:
    print(f"❌ Error: {news_data.get('error', 'Unknown error')}")

✅ Success: True
📰 Symbol: AAPL
📊 Total articles: 5

1. I want to pay my late 60s parents $1,000 monthly for babysitting. They refuse the money: should I invest it for them instead?
   Source: Yahoo Finance | Date: 2026-05-18T16:57:50Z
   Summary: A listener named Carrie Anne wrote into the Rich Habits Podcast with a problem most adult children w...

2. SpaceX’s Mega-IPO Puts a Price on the Fear of Missing Out
   Source: Yahoo Finance | Date: 2026-05-18T16:55:43Z
   Summary: (Bloomberg) -- Elon Musk has turned science fiction into reality for the better part of two decades,...

3. Intel Apple Chip Pact Tests Valuation As U.S. Foundry Ambitions Grow
   Source: Yahoo Finance | Date: 2026-05-18T16:17:40Z
   Summary: Intel (NasdaqGS:INTC) and Apple have reached a preliminary agreement for U.S.-based chip production....



In [17]:
# Cell: Define Financial Expert response models
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime

class KeyFinding(BaseModel):
    """Individual finding from analysis"""
    category: str  # e.g., "Fundamentals", "Technical", "News Sentiment"
    finding: str
    impact: str  # "Positive", "Negative", "Neutral"

class RiskFactor(BaseModel):
    """Risk factor identified"""
    risk: str
    severity: str  # "Low", "Medium", "High"
    mitigation: Optional[str] = None

class ExpertAnalysisResponse(BaseModel):
    """Structured response from Financial Expert Agent"""
    success: bool
    symbol: str
    recommendation: str  # "Strong Buy", "Buy", "Hold", "Sell", "Strong Sell"
    confidence: float  # 0.0 to 1.0
    target_price: Optional[float] = None
    current_price: Optional[float] = None
    
    # Analysis components
    key_findings: List[KeyFinding]
    risk_factors: List[RiskFactor]
    summary: str
    
    # Metadata
    analyzed_at: str = Field(default_factory=lambda: datetime.now().isoformat())
    error: Optional[str] = None

print("✅ Expert response models defined")

✅ Expert response models defined


In [18]:
# System prompt for the expert agent
expert_system_prompt = """You are a Financial Expert Agent specializing in stock analysis.

Your job is to analyze stock data and news to provide investment recommendations.

Input you will receive:
- Stock data: current price, historical performance, P/E ratio, market cap
- News articles: recent headlines and summaries about the stock

Your analysis should include:
1. **Recommendation**: Strong Buy, Buy, Hold, Sell, or Strong Sell
2. **Confidence**: A score from 0.0 to 1.0 based on data quality and clarity
3. **Key Findings**: 3-5 important observations from data and news
4. **Risk Factors**: Potential risks to consider
5. **Summary**: Concise conclusion with reasoning

Consider these factors:
- Price trends (momentum, support/resistance)
- Valuation (P/E relative to sector)
- News sentiment (positive/negative developments)
- Market conditions

Return ONLY structured data in the required format. Do not add extra text outside the JSON."""

# Create the agent (no tools needed for now, just analysis)
financial_expert_agent = create_agent(
    model=llm,
    system_prompt=expert_system_prompt,
    response_format=ExpertAnalysisResponse,
    name="financial_expert_agent"
)

print("✅ Financial Expert Agent created")

✅ Financial Expert Agent created


In [19]:
from typing import List, Optional, Literal

In [20]:

from typing import Literal, List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END, START
from langgraph.types import Command
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI

# Define the state
class MultiAgentState(TypedDict):
    messages: List[Any]
    stock_symbol: str
    data_result: Any
    news_result: Any
    expert_analysis: Any
    next: str
    iteration: int

members = ["data_fetcher", "news_agent", "financial_expert"]

def make_supervisor_node(llm, members: List[str]):
    """Create a supervisor node with rate limit handling"""
    
    options = ["FINISH"] + members
    system_prompt = f"""
    You are a supervisor tasked with managing a conversation between the following workers: {members}.
    
    Your role:
    - For stock analysis, follow this sequence: data_fetcher → news_agent → financial_expert → FINISH
    - Never skip steps - always get data before news, and both before analysis
    - Never do the analysis yourself - always delegate to financial_expert
    - When all tasks are complete, respond with FINISH
    
    Respond with the worker to act next. When finished, respond with FINISH.
    """
    
    class Router(TypedDict):
        next: Literal[*options]
    
    # ساخت structured LLM یک بار
    structured_llm = llm.with_structured_output(Router)
    
    def supervisor_node(state: MultiAgentState) -> Command:
        """LLM-based router with rate limit handling"""
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Current state: Data fetched: {state.get('data_result') is not None}, News fetched: {state.get('news_result') is not None}, Analysis done: {state.get('expert_analysis') is not None}"}
        ]
        
        for msg in state.get("messages", []):
            if isinstance(msg, HumanMessage):
                messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                messages.append({"role": "assistant", "content": msg.content})
        
        # Cell: تابع هندل ریتریت برای هر چی (هم invoke معمولی هم with_structured_output)


        def call_with_retry(func, *args, max_retries=10, **kwargs):
            """
            هندل ریتریت برای هر نوع تابعی - چه invoke معمولی چه with_structured_output
            """
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if "rate limit" in str(e).lower() or "429" in str(e):
                        wait = 5 * (attempt + 1)
                        print(f"⚠️ Rate limit! Waiting {wait} seconds... (attempt {attempt + 1}/{max_retries})")
                        time.sleep(wait)
                    else:
                        raise e
            raise Exception(f"Still failing after {max_retries} retries")
        # اینجا از call_with_retry استفاده کن
        response = call_with_retry(structured_llm.invoke, messages)
        goto = response["next"]
        
        if goto == "FINISH":
            goto = END
        
        return Command(goto=goto, update={"next": goto, "iteration": state.get("iteration", 0) + 1})
    
    return supervisor_node

print("✅ Supervisor node factory with rate limit handling created")

✅ Supervisor node factory with rate limit handling created


In [21]:
# Cell: Data Fetcher Agent Node
def data_fetcher_node(state: MultiAgentState) -> Command[Literal["supervisor"]]:
    """
    Data Fetcher agent node that fetches stock data and returns to supervisor
    """
    symbol = state.get("stock_symbol", "")
    

    # Prepare the input for the agent
    input_state = {
        "messages": [("user", f"Get current and historical stock data for {symbol}")]
    }
    
    # Invoke the agent
    result = call_with_retry(
        data_fetcher_agent, 
        {"messages": [("user", f"Get stock data for {symbol}")]}
    )
    
    # Extract the response content
    response_content = result["messages"][-1].content
    
    # Parse JSON to structured data if needed
    try:
        import json
        if isinstance(response_content, str):
            data_result = json.loads(response_content)
        else:
            data_result = response_content
    except:
        data_result = {"success": False, "error": "Failed to parse", "raw": str(response_content)}
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=f"✅ Stock data retrieved for {symbol}", name="data_fetcher")
            ],
            "data_result": data_result,
            "iteration": state.get("iteration", 0) + 1
        },
        goto="supervisor"
    )

print("✅ Data fetcher node created")

✅ Data fetcher node created


In [22]:
# Cell: News Agent Node
def news_agent_node(state: MultiAgentState) -> Command[Literal["supervisor"]]:
    """
    News Agent node that fetches news and returns to supervisor
    """
    symbol = state.get("stock_symbol", "")
    
    # Prepare input
    input_state = {
        "messages": [("user", f"Get recent news for {symbol}")]
    }
    
    # Invoke the agent
    result = call_with_retry(
        news_agent,
        {"messages": [("user", f"Get news for {symbol}")]}
    )
    
    # Extract response
    response_content = result["messages"][-1].content
    
    # Parse JSON
    try:
        import json
        if isinstance(response_content, str):
            news_result = json.loads(response_content)
        else:
            news_result = response_content
    except:
        news_result = {"success": False, "error": "Failed to parse", "raw": str(response_content)}
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=f"📰 News fetched for {symbol}", name="news_agent")
            ],
            "news_result": news_result,
            "iteration": state.get("iteration", 0) + 1
        },
        goto="supervisor"
    )

print("✅ News agent node created")

✅ News agent node created


In [23]:
# Cell: Financial Expert Node
def financial_expert_node(state: MultiAgentState) -> Command[Literal["supervisor"]]:
    """
    Financial Expert node that analyzes data and returns to supervisor
    """
    symbol = state.get("stock_symbol", "")
    data_result = state.get("data_result", {})
    news_result = state.get("news_result", {})
    
    # Prepare context with previous results
    context = f"""
    Analyze {symbol} stock and provide investment recommendation.
    
    STOCK DATA:
    {data_result}
    
    NEWS:
    {news_result}
    
    Based on this information, provide:
    1. Recommendation (Strong Buy/Buy/Hold/Sell/Strong Sell)
    2. Confidence level (0-1)
    3. Key findings
    4. Risk factors
    5. Summary
    """
    
    
    # Invoke the agent
    input_state = {
        "messages": [("user", context)]
    }
    
    result = call_with_retry(
        financial_expert_agent,
        {"messages": [("user", context)]}
    )
    
    # Extract response
    response_content = result["messages"][-1].content
    
    # Parse JSON
    try:
        import json
        if isinstance(response_content, str):
            analysis_result = json.loads(response_content)
        else:
            analysis_result = response_content
    except:
        analysis_result = {"success": False, "error": "Failed to parse", "raw": str(response_content)}
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=f"💡 Analysis completed for {symbol}", name="financial_expert")
            ],
            "expert_analysis": analysis_result,
            "iteration": state.get("iteration", 0) + 1
        },
        goto="supervisor"
    )

print("✅ Financial expert node created")

✅ Financial expert node created


In [24]:
supervisor = make_supervisor_node(llm=llm , members= ["data","new","analysis"])

In [25]:
# Cell: Build the graph with all nodes
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# Create the graph builder
builder = StateGraph(MultiAgentState)

# Add all nodes
builder.add_node("supervisor", supervisor)
builder.add_node("data", data_fetcher_node)
builder.add_node("new", news_agent_node)
builder.add_node("analysis", financial_expert_node)

# Add edges - from supervisor to agents
builder.add_edge(START, "supervisor")

# Each agent returns to supervisor via Command, so no explicit edges needed

# Compile the graph
app = builder.compile(checkpointer=InMemorySaver())

print("✅ Multi-agent system compiled!")
print("\nGraph structure:")
print("  START → supervisor → (data_fetcher → supervisor → news_agent → supervisor → financial_expert → supervisor → END)")
app

✅ Multi-agent system compiled!

Graph structure:
  START → supervisor → (data_fetcher → supervisor → news_agent → supervisor → financial_expert → supervisor → END)


ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [40]:
# Cell: Test the multi-agent system
from datetime import datetime

# Initial state
initial_state = {
    "messages": [],
    "stock_symbol": "AAPL",
    "data_result": None,
    "news_result": None,
    "expert_analysis": None,
    "next": "",
    "iteration": 0
}

print("🚀 Starting multi-agent analysis...")
print("="*60)

# Run the graph
final_state = app.invoke(
    initial_state,
    config={"configurable": {"thread_id": "analysis_1"}}
)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

# Display final analysis
if final_state.get("expert_analysis"):
    analysis = final_state["expert_analysis"]
    if isinstance(analysis, dict):
        print(f"\n🎯 Recommendation: {analysis.get('recommendation', 'N/A')}")
        print(f"📊 Confidence: {analysis.get('confidence', 0) * 100:.0f}%")
        print(f"\n💡 Summary: {analysis.get('summary', 'N/A')[:300]}")
    else:
        print(f"\n📊 Analysis: {analysis}")
else:
    print("\n❌ No analysis available")

print(f"\n✅ Total iterations: {final_state.get('iteration', 0)}")

🚀 Starting multi-agent analysis...


KeyboardInterrupt: 

In [27]:
# Initial state
initial_state = {
    "messages": [],
    "stock_symbol": "AAPL",
    "data_result": None,
    "news_result": None,
    "expert_analysis": None,
    "next": "",
    "iteration": 0
}

print("🚀 Starting multi-agent analysis...")
print("="*60)
# Cell: Stream the execution to see the flow
print("🔄 Streaming execution...")
print("="*60)

for event in app.stream(
    initial_state,
    config={"configurable": {"thread_id": "analysis_2"}},
    stream_mode="values"
):
    if "next" in event and event["next"]:
        print(f"📍 Router → {event['next']}")
    
    if "messages" in event and event["messages"]:
        for msg in event["messages"]:
            if hasattr(msg, "name") and msg.name:
                print(f"✅ {msg.name.upper()}: {msg.content}")

print("\n" + "="*60)
print("✅ Execution complete!")

🚀 Starting multi-agent analysis...
🔄 Streaming execution...
📍 Router → data
📍 Router → data
✅ DATA_FETCHER: ✅ Stock data retrieved for AAPL
⚠️ Rate limit! Waiting 5 seconds... (attempt 1/10)


APIConnectionError: Connection error.